In [1]:
import json
import os
import pandas as pd


In [2]:
# -------- OUTILS GÉNÉRIQUES --------------
def load_json(path):
    if not os.path.exists(path):
        print(f" Fichier introuvable → {path}")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


In [3]:

# -------- TEST INTÉGRITÉ ------------------

def test_integrity(label, records):
    print(f"\n TEST INTÉGRITÉ → {label}")

    if not records or len(records) == 0:
        print(" Aucune donnée")
        return

    df = pd.DataFrame(records)

    print(f" Nombre d’enregistrements : {len(df)}")

    # Colonnes
    print(" Colonnes :", list(df.columns))

    # Types
    print("\n Types détectés :")
    print(df.dtypes)

    # Valeurs manquantes
    missing = df.isna().sum()
    print("\n Valeurs manquantes :")
    print(missing[missing > 0] if missing.sum() else "RAS")

    # Doublons
    dups = df.duplicated().sum()
    print(f"\n Doublons : {dups if dups else 'Aucun'}")


In [4]:
# -------- TRANSFORMATION STATIONS --------

def normalize_measure(rec):
    return {
        "timestamp": rec.get("date") or rec.get("time"),
        "temperature": rec.get("temp") or rec.get("temperature"),
        "humidity": rec.get("humidity"),
        "pressure": rec.get("pressure"),
    }


def build_personal_doc(station_id, measures):
    return {
        "_id": station_id,
        "provider": "personal_station",
        "measures": measures
    }


def process_personal_station(file, station_id, out_name):
    print(f"\n---- Traitement station personnelle → {station_id} ----")

    raw = load_json(file)
    if raw is None:
        return

    measures_raw = raw.get("data", raw) if isinstance(raw, dict) else raw

    # Tests avant
    test_integrity(f"{station_id} — AVANT", measures_raw)

    # Transformation
    measures = [normalize_measure(r) for r in measures_raw]

    # Tests après
    test_integrity(f"{station_id} — APRÈS", measures)

    doc = build_personal_doc(station_id, measures)
    save_json(out_name, doc)

    print(f" Sauvé → {out_name}")


In [5]:
# -------- INFOCLIMAT ----------------------

def process_infoclimat(file):
    print("\n---- Traitement InfoClimat ----")

    raw = load_json(file)
    if raw is None:
        return

    stations_raw = raw.get("stations", [])

    # Tests avant
    test_integrity("InfoClimat — AVANT", stations_raw)

    docs = []
    for s in stations_raw:
        docs.append({
            "_id": s.get("id"),
            "provider": "infoclimat",
            "name": s.get("name"),
            "lat": s.get("lat"),
            "lon": s.get("lon"),
            "measures": s.get("data", [])
        })

    # Test après
    test_integrity("InfoClimat — APRÈS", docs)

    save_json("infoclimat_mongo.json", docs)
    print(" Sauvé → infoclimat_mongo.json")


In [6]:
# ---------------- MAIN --------------------


if __name__ == "__main__":

    print("===== TRANSFORMATION + TESTS =====")

    process_personal_station(
        "data/la_madeleine.json",
        "ILAMAD25",
        "la_madeleine_mongo.json"
    )

    process_personal_station(
        "data/ichtegem.json",
        "IICHTE19",
        "ichtegem_mongo.json"
    )

    process_infoclimat(
        "data/InfoClimat.json"
    )

    print("\n FINI — 3 fichiers traités + tests OK")


===== TRANSFORMATION + TESTS =====

---- Traitement station personnelle → ILAMAD25 ----

 TEST INTÉGRITÉ → ILAMAD25 — AVANT
 Nombre d’enregistrements : 1915
 Colonnes : ['station_id', 'date', 'time', 'temperature', 'humidity', 'pressure', 'wind_speed', 'wind_gust', 'uv', 'solar']

 Types détectés :
station_id      object
date            object
time            object
temperature    float64
humidity       float64
pressure       float64
wind_speed     float64
wind_gust      float64
uv             float64
solar          float64
dtype: object

 Valeurs manquantes :
time           7
temperature    7
humidity       7
pressure       7
wind_speed     7
wind_gust      7
uv             7
solar          7
dtype: int64

 Doublons : Aucun

 TEST INTÉGRITÉ → ILAMAD25 — APRÈS
 Nombre d’enregistrements : 1915
 Colonnes : ['timestamp', 'temperature', 'humidity', 'pressure']

 Types détectés :
timestamp       object
temperature    float64
humidity       float64
pressure       float64
dtype: object

 Vale

TypeError: unhashable type: 'dict'